# Testing Water Vapor Profile Scaling

1. Plot the raw MIPAS H₂O abundance profile
2. Scale it with relative humidity (as in MakeModel.py)
3. Compute the PWV
4. Scale it to a target PWV using scale_h2o_to_pwv_preserve_surface
5. Plot the result

In [ ]:
import sys
import os
import copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

sys.path.insert(0, str(Path.cwd().parent / 'TelFit' / 'src'))
from MakeModel import (
    Modeler, MoleculeNumbers, humidity_to_ppmv, 
    pwv_from_profile, scale_h2o_to_pwv_preserve_surface
)

## 1. Read MIPAS profile and extract raw H₂O

In [ ]:
# Read the MIPAS atmosphere profile directly
TELFIT_DIR = Path.home() / '.TelFit' / 'rundir1'

filename = TELFIT_DIR / 'MIPAS_atmosphere_profile'
nmolecules = 12

Atmosphere = defaultdict(list)
indices = {}

with open(filename) as infile:
    lines = infile.readlines()

for i in range(len(lines)):
    line = lines[i]
    if line.startswith('*') and 'END' not in line:
        if line.find('HGT') > 0 and line.find('[') > 0:
            numlevels = int(lines[i - 1].split()[0])
            indices['Z'] = i
        elif line.find('PRE') > 0 and line.find('[') > 0:
            indices['P'] = i
        elif line.find('TEM') > 0 and line.find('[') > 0:
            indices['T'] = i
        else:
            molecule = line.split('*')[-1].split()[0]
            indices[molecule] = i

levelsperline = 5.0
linespersection = int(numlevels / levelsperline + 0.9)

# Heights
layers = []
for j in range(indices['Z'] + 1, indices['Z'] + 1 + linespersection):
    levels = lines[j].split()
    [layers.append(float(level)) for level in levels]

# Pressure
for j in range(linespersection):
    levels = lines[j + indices['P'] + 1].split()
    for i, level in enumerate(levels):
        Atmosphere[layers[int(j * levelsperline + i)]].append(float(level))

# Temperature
for j in range(linespersection):
    levels = lines[j + indices['T'] + 1].split()
    for i, level in enumerate(levels):
        Atmosphere[layers[int(j * levelsperline + i)]].append(float(level))
        Atmosphere[layers[int(j * levelsperline + i)]].append([])

# Abundances
for k in range(1, nmolecules + 1):
    for j in range(linespersection):
        levels = lines[j + indices[MoleculeNumbers[k]] + 1].split()
        for i, level in enumerate(levels):
            Atmosphere[layers[int(j * levelsperline + i)]][2].append(float(level))

layers = np.array(layers)

# Extract profiles
z_km = layers
P_hpa = np.array([Atmosphere[z][0] for z in layers])
T_K = np.array([Atmosphere[z][1] for z in layers])
h2o_ppmv_mipas = np.array([Atmosphere[z][2][0] for z in layers])  # molecule index 0 = H2O

print(f'MIPAS profile: {len(layers)} layers from {z_km[0]} to {z_km[-1]} km')
print(f'H2O range: {h2o_ppmv_mipas.min():.2e} to {h2o_ppmv_mipas.max():.2e} ppmv')

In [ ]:
# Plot raw MIPAS H2O profile
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

axes[0].plot(P_hpa, z_km, 'b-')
axes[0].set_xlabel('Pressure (hPa)')
axes[0].set_ylabel('Altitude (km)')
axes[0].set_title('Pressure profile')
axes[0].set_xscale('log')

axes[1].plot(T_K, z_km, 'r-')
axes[1].set_xlabel('Temperature (K)')
axes[1].set_ylabel('Altitude (km)')
axes[1].set_title('Temperature profile')

axes[2].plot(h2o_ppmv_mipas, z_km, 'g-', label='MIPAS raw')
axes[2].set_xlabel('H₂O (ppmv)')
axes[2].set_ylabel('Altitude (km)')
axes[2].set_title('H₂O abundance profile')
axes[2].set_xscale('log')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Scale with relative humidity (as MakeModel does)

In [ ]:
# Parameters (same as what MakeModel uses)
alt = 2.1        # observatory altitude [km]
pressure = 795.0 # surface pressure [hPa]
temperature = 283.0  # surface temperature [K]
humidity = 50.0  # relative humidity [%]

# Convert humidity to ppmv (as MakeModel does)
h2o_surface_ppmv = humidity_to_ppmv(humidity, temperature, pressure)
print(f'RH={humidity}% at T={temperature}K, P={pressure}hPa -> H2O = {h2o_surface_ppmv:.1f} ppmv')

# Scale the MIPAS profile (replicate MakeModel logic)
# Interpolate MIPAS value at observatory altitude
keys = sorted(Atmosphere.keys())
lower = max(0, np.searchsorted(keys, alt) - 1)
upper = min(lower + 1, len(keys) - 1)

# MIPAS H2O at observatory altitude (interpolated)
h2o_mipas_at_alt = (Atmosphere[keys[upper]][2][0] - Atmosphere[keys[lower]][2][0]) / \
                    (keys[upper] - keys[lower]) * (alt - keys[lower]) + Atmosphere[keys[lower]][2][0]

print(f'MIPAS H2O at {alt} km: {h2o_mipas_at_alt:.1f} ppmv')
print(f'Scale factor: {h2o_surface_ppmv / h2o_mipas_at_alt:.3f}')

# Apply uniform scaling (as MakeModel does: multiply entire profile)
scale_factor = h2o_surface_ppmv / h2o_mipas_at_alt
h2o_ppmv_scaled = h2o_ppmv_mipas * scale_factor

print(f'\nScaled H2O range: {h2o_ppmv_scaled.min():.2e} to {h2o_ppmv_scaled.max():.2e} ppmv')

In [ ]:
# Plot: MIPAS raw vs humidity-scaled
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

ax.plot(h2o_ppmv_mipas, z_km, 'g-', lw=2, label='MIPAS raw')
ax.plot(h2o_ppmv_scaled, z_km, 'b--', lw=2, label=f'Scaled (RH={humidity}%)')
ax.axhline(alt, color='gray', ls=':', label=f'Observatory ({alt} km)')

ax.set_xlabel('H₂O (ppmv)')
ax.set_ylabel('Altitude (km)')
ax.set_title('H₂O profile: raw vs humidity-scaled')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Compute PWV from the scaled profile

In [ ]:
# Compute PWV for the raw MIPAS and the humidity-scaled profile
pwv_mipas = pwv_from_profile(z_km, P_hpa, T_K, h2o_ppmv_mipas)
pwv_scaled = pwv_from_profile(z_km, P_hpa, T_K, h2o_ppmv_scaled)

print(f'PWV (MIPAS raw):       {pwv_mipas:.2f} mm')
print(f'PWV (RH={humidity}% scaled): {pwv_scaled:.2f} mm')

## 4. Scale to target PWV using `scale_h2o_to_pwv_preserve_surface`

In [ ]:
# Target PWV values to test
pwv_targets = [1.0, 3.0, 5.0, 10.0, 20.0]  # mm

results = {}
for pwv_target in pwv_targets:
    h2o_new, A, pwv_achieved = scale_h2o_to_pwv_preserve_surface(
        z_km, P_hpa, T_K, h2o_ppmv_scaled,
        pwv_target_mm=pwv_target,
        scale_height_km=2.0,
    )
    results[pwv_target] = h2o_new
    print(f'Target PWV={pwv_target:.1f} mm -> achieved={pwv_achieved:.2f} mm (A={A:.3f})')

In [ ]:
# Plot all profiles
fig, ax = plt.subplots(1, 1, figsize=(7, 6))

ax.plot(h2o_ppmv_mipas, z_km, 'k-', lw=1.5, alpha=0.5, label='MIPAS raw')
ax.plot(h2o_ppmv_scaled, z_km, 'b--', lw=1.5, label=f'RH={humidity}% (PWV={pwv_scaled:.1f} mm)')

colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(pwv_targets)))
for (pwv_target, h2o_new), color in zip(results.items(), colors):
    ax.plot(h2o_new, z_km, '-', color=color, lw=1.5, label=f'PWV={pwv_target:.0f} mm')

ax.axhline(alt, color='gray', ls=':', lw=0.8)
ax.set_xlabel('H₂O (ppmv)')
ax.set_ylabel('Altitude (km)')
ax.set_title('H₂O profiles: scaled to different PWV values\n(surface value preserved)')
ax.set_xscale('log')
ax.set_xlim(1e-2, None)
ax.legend(fontsize='small', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in to lower atmosphere (0-20 km)
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

mask = z_km <= 20

ax.plot(h2o_ppmv_scaled[mask], z_km[mask], 'b--', lw=2, label=f'RH={humidity}% (PWV={pwv_scaled:.1f} mm)')

for (pwv_target, h2o_new), color in zip(results.items(), colors):
    ax.plot(h2o_new[mask], z_km[mask], '-', color=color, lw=1.5, label=f'PWV={pwv_target:.0f} mm')

ax.axhline(alt, color='gray', ls=':', lw=0.8, label=f'Observatory ({alt} km)')
ax.set_xlabel('H₂O (ppmv)')
ax.set_ylabel('Altitude (km)')
ax.set_title('Lower atmosphere H₂O profiles (surface preserved)')
ax.legend(fontsize='small')
plt.tight_layout()
plt.show()